# Build the Music Database

This notebook transforms the cleaned music dataset into a **normalized SQLite database** for downstream SQL analysis.

## What this notebook does

1. Load the cleaned CSV dataset with pandas.
2. Create a stable `track_key` used as the internal database key.
3. Split the source data into separate tables for metadata, audio features, and genres.
4. Normalize the multi-value `all_genres` column into the `track_genres` table.
5. Rebuild the SQLite schema with primary keys, foreign keys, constraints, and useful indexes.
6. Load the prepared DataFrames into SQLite in the correct foreign-key order.

### Resulting schema

```text
tracks_metadata
      │
      ├────────── 1:1 ────────── audio_features
      │
      └────────── 1:N ────────── track_genres
```

`tracks_metadata.track_key` is the parent key. Both child tables reference it with foreign keys.

> **Important:** Running the notebook rebuilds the database tables. It is designed to be rerunnable (idempotent): the existing tables are dropped and recreated before the data is loaded.

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

## 1. Imports and file paths

The notebook uses:

- **SQLite** for the relational database.
- **pandas** for loading and transforming the CSV data.
- **Pathlib** for filesystem paths.

`DB` is the output SQLite database, while `CSV` is the source dataset.

In [2]:
DB = Path("music.db")
CSV = Path("data/CleanedData.csv")

## 2. Load the source data and create the internal key

`get_conn()` centralizes the SQLite connection settings:

- `foreign_keys = ON` enables foreign-key enforcement for each connection.
- `journal_mode = WAL` uses SQLite's write-ahead logging mode.
- `busy_timeout = 5000` gives SQLite up to 5 seconds when the database is temporarily locked.

The CSV index is reset and stored as `track_key`. This key is used internally by the database to connect the metadata, audio-feature, and genre records.

In [3]:
# PRAGMA foreign_keys is per-connection, not stored in the file.
# This helper must be used for every connection.

def get_conn(path: Path) -> sqlite3.Connection:
    conn = sqlite3.connect(path)
    conn.execute("PRAGMA foreign_keys = ON;")   # must be outside a transaction
    conn.execute("PRAGMA journal_mode = WAL;")
    conn.execute("PRAGMA busy_timeout = 5000;")
    return conn


df = pd.read_csv(CSV)
df = df.reset_index(drop=True)
df["track_key"] = df.index.astype(int)

## 3. Define the database table columns

The source DataFrame is split into two main groups:

- **Metadata:** identifying and descriptive information about each track.
- **Audio features:** numerical properties such as energy, valence, tempo, and danceability.

The genre information is handled separately below because `all_genres` can contain multiple genres for one track.

In [4]:
# Split frames
metadata_cols = [
    "track_key", "track_id", "track_name", "artists",
    "album_name", "track_genre", "popularity", "duration_m", "explicit",
]
audio_cols = [
    "track_key", "danceability", "energy", "key", "loudness", "mode",
    "speechiness", "acousticness", "instrumentalness",
    "liveness", "valence", "tempo", "time_signature",
]

## 4. Prepare the DataFrames for SQLite

The source column names are adjusted to make their database meaning clearer:

- `track_id` → `spotify_track_id`
- `track_genre` → `primary_genre`

Integer-like columns are explicitly converted to integers so SQLite receives them as `INTEGER` rather than `REAL`.

In [5]:
df_meta = df[metadata_cols].rename(
    columns={"track_id": "spotify_track_id", "track_genre": "primary_genre"}
)
df_audio = df[audio_cols].copy()

# Cast integer-like columns to Python int so SQLite stores INTEGER, not REAL
for col in ["popularity", "explicit"]:
    df_meta[col] = df_meta[col].astype(int)
for col in ["key", "mode", "time_signature"]:
    df_audio[col] = df_audio[col].astype(int)

## 5. Normalize the multi-value genre column

A track can have multiple values in `all_genres`, separated by semicolons. Storing all of those values in one column would make genre-based SQL queries harder.

Here, the values are split and **exploded** into one row per track–genre relationship. The resulting `genres` DataFrame becomes the `track_genres` junction table.

For example:

| track_key | all_genres | → | track_genre |
|---:|---|---|---|
| 42 | `pop;dance;electronic` | → | `pop` |
| 42 | `pop;dance;electronic` | → | `dance` |
| 42 | `pop;dance;electronic` | → | `electronic` |

Duplicate track–genre pairs are removed before loading the table.

In [6]:
# explode all_genres into a proper 1:N junction table
genres = (
    df[["track_key", "all_genres"]]
    .assign(track_genre=lambda d: d["all_genres"].str.split(";"))
    .explode("track_genre")
    .loc[lambda d: d["track_genre"].ne("")]
    [["track_key", "track_genre"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

## 6. Create the SQLite schema

The tables are created in parent/child order conceptually, but existing child tables are dropped first so the database can be rebuilt safely.

### `tracks_metadata`
One row per track. `track_key` is the primary key and `spotify_track_id` must be unique and non-null.

### `audio_features`
One row per track. Its `track_key` is both the primary key and a foreign key to `tracks_metadata`, creating a 1:1 relationship.

### `track_genres`
One row per track–genre relationship. Its composite primary key (`track_key`, `track_genre`) prevents duplicate relationships.

### Constraints and indexes
The schema also adds `CHECK` constraints for values such as popularity, mode, key, and time signature. Indexes are added for artist and genre lookups to support common downstream filters.

In [7]:
# idempotent rebuild — drop child tables first, then parent.
conn = get_conn(DB)
try:
    with conn:
        conn.execute("DROP TABLE IF EXISTS track_genres;")
        conn.execute("DROP TABLE IF EXISTS audio_features;")
        conn.execute("DROP TABLE IF EXISTS tracks_metadata;")

        conn.execute("""
        CREATE TABLE tracks_metadata (
            track_key        INTEGER PRIMARY KEY,
            spotify_track_id TEXT UNIQUE NOT NULL,
            track_name       TEXT,
            artists          TEXT,
            album_name       TEXT,
            primary_genre    TEXT,
            popularity       INTEGER CHECK (popularity BETWEEN 0 AND 100),
            duration_m       REAL,
            explicit         INTEGER CHECK (explicit IN (0, 1))
        );
        """)

        conn.execute("""
        CREATE TABLE audio_features (
            track_key        INTEGER PRIMARY KEY,
            danceability     REAL,
            energy           REAL,
            key              INTEGER CHECK (key BETWEEN -1 AND 11),
            loudness         REAL,
            mode             INTEGER CHECK (mode IN (0, 1)),
            speechiness      REAL,
            acousticness     REAL,
            instrumentalness REAL,
            liveness         REAL,
            valence          REAL,
            tempo            REAL,
            time_signature   INTEGER CHECK (time_signature BETWEEN 3 AND 7),
            FOREIGN KEY (track_key) REFERENCES tracks_metadata(track_key)
                ON DELETE CASCADE
        );
        """)

        conn.execute("""
        CREATE TABLE track_genres (
            track_key   INTEGER NOT NULL,
            track_genre TEXT    NOT NULL,
            PRIMARY KEY (track_key, track_genre),
            FOREIGN KEY (track_key) REFERENCES tracks_metadata(track_key)
                ON DELETE CASCADE
        );
        """)

        # FIX #9: indexes for the obvious downstream filters
        conn.execute("CREATE INDEX idx_tracks_artists ON tracks_metadata(artists);")
        conn.execute("CREATE INDEX idx_track_genres_genre ON track_genres(track_genre);")
finally:
    conn.close()

## 7. Load the transformed data

The DataFrames are inserted into SQLite using `append` because the tables have already been created with the intended schema.

The load order matters because `audio_features` and `track_genres` reference `tracks_metadata` through foreign keys. The parent table is therefore populated first.

In [8]:
# Load — parent first, then children (FK order matters)
conn = get_conn(DB)
try:
    with conn:
        df_meta.to_sql("tracks_metadata", conn, if_exists="append", index=False)
        df_audio.to_sql("audio_features", conn, if_exists="append", index=False)
        genres.to_sql("track_genres", conn, if_exists="append", index=False)
finally:
    conn.close()

## Database ready

After the final cell runs successfully, `music.db` contains the normalized tables and indexes defined above.

The database is now ready for SQL analysis, such as genre-level aggregations, track filtering, and joins between metadata and audio features.